In [11]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [12]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_juni
Connected to new database: dataleap_v5_migration
Connected to future database: db_future


In [13]:
import pickle
import os

print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 ")
print("================================================================================")

# ================================================================================
# KONFIGURASI 1: LOAD MASING-MASING FILE PICKLE (TETAP TERPISAH)
# ================================================================================
data_cimut = {}
data_afrida = {}
data_hanif = {}

# 1. Load File cimut (Ganti nama file sesuai punyamu)
try:
    with open('fase_4_cimut.pkl', 'rb') as f:
        data_cimut = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik cimut.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_4_afrida.pkl', 'rb') as f:
        data_afrida = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Afrida.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif
try:
    with open('fase_4_hanif.pkl', 'rb') as f:
        data_hanif = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Hanif.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Hanif: {e}")

print("\n================================================================================")
# ================================================================================
# KONFIGURASI 2: ISI DAFTAR TABEL MILIK MASING-MASING ORANG
# ================================================================================
list_table_cimut = [
    "izin_karyawan",
    "verifikasi_izin",
    "absensi",
    "verifikasi_absensi",
    "karyawan_resign",
]

list_table_afrida = [
    "jadwal",
    "jadwal_hari",
    "jadwal_detail",
    "jadwal_pengajar",
    "jadwal_siswa",
    "catatan_kelas",
    "catatan_kelas_tag",
    "catatan_mingguan",
]

list_table_hanif = [
    "siswa",
    "kursus_siswa",
    "siswa_keluar",
    "mitra",
    "mitra_progres",
    "kemitraan_verifikator",
    "siswa_mitra",
    "siswa_mitra_keluar",
]

# ================================================================================
# KONFIGURASI 3: ATUR URUTAN MUTLAK PENYUNTIKAN KE DATABASE (MASTER ORDER)
# ================================================================================
# Masukkan nama tabel yang mau di-insert sesuai urutan FK (Foreign Key).
# Kamu bebas menyilangkan nama tabel di sini, sistem akan otomatis mencari pemiliknya.
master_urutan_insert = [
    "izin_karyawan",
    "verifikasi_izin",
    "absensi",
    "verifikasi_absensi",
    "karyawan_resign",
    "jadwal",
    "jadwal_hari",
    "jadwal_detail",
    "jadwal_pengajar",
    "jadwal_siswa",
    "catatan_kelas",
    "catatan_kelas_tag",
    "catatan_mingguan",
    "siswa",
    "kursus_siswa",
    "siswa_keluar",
    "mitra",
    "mitra_progres",
    "kemitraan_verifikator",
    "siswa_mitra",
    "siswa_mitra_keluar",
]

# ================================================================================
# SISTEM DETEKTIF: MENCARI DAN MENGGABUNGKAN DATA BERDASARKAN PEMILIKNYA
# ================================================================================
print("🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...\n")

data_siap_insert = {}

for table in master_urutan_insert:
    if table in list_table_cimut:
        data_siap_insert[table] = data_cimut.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data cimut.")
        
    elif table in list_table_afrida:
        data_siap_insert[table] = data_afrida.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Afrida.")
        
    elif table in list_table_hanif:
        data_siap_insert[table] = data_hanif.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Hanif.")
        
    else:
        # Jika kamu memasukkan nama tabel di master_urutan tapi lupa memasukkannya di list pemilik
        data_siap_insert[table] = None
        print(f"  ❌ ERROR: Tabel '{table}' tidak ada di list cimut, Afrida, maupun Hanif!")

print("\n✅ Pemetaan selesai! Data siap disuntikkan ke database.")

 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 


✓ Berhasil memuat data PKL milik cimut.
✓ Berhasil memuat data PKL milik Afrida.
⚠️ Peringatan: Gagal memuat file pkl Hanif: (str, array(['2022-07-01', '2022-07-01', '2021-07-01', ..., '2026-05-26',
       '2026-06-04', '2026-06-05'], shape=(1500,), dtype=object))

🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...

  📦 Tabel 'izin_karyawan' otomatis dipetakan dari data cimut.
  📦 Tabel 'verifikasi_izin' otomatis dipetakan dari data cimut.
  📦 Tabel 'absensi' otomatis dipetakan dari data cimut.
  📦 Tabel 'verifikasi_absensi' otomatis dipetakan dari data cimut.
  📦 Tabel 'karyawan_resign' otomatis dipetakan dari data cimut.
  📦 Tabel 'jadwal' otomatis dipetakan dari data Afrida.
  📦 Tabel 'jadwal_hari' otomatis dipetakan dari data Afrida.
  📦 Tabel 'jadwal_detail' otomatis dipetakan dari data Afrida.
  📦 Tabel 'jadwal_pengajar' otomatis dipetakan dari data Afrida.
  📦 Tabel 'jadwal_siswa' otomatis dipetakan dari data Afrida.
  📦 Tabel 'catatan_kelas' otomatis dipetakan dari dat

## Hide code

In [ ]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT (ANTI SILENT-KILLER, AUTO-BATCHING & DIAGNOSTIC)
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list, batch_size=2000):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DENGAN CHUNKING (LOOPING AMAN)
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {'status': 'not_found', 'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl', 'warnings': []}
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {'status': 'empty', 'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)', 'warnings': []}
            continue
            
        try:
                        # Bersihkan kolom kosong murni
               # Bersihkan kolom kosong murni
            df_to_push = df_target.dropna(axis=1, how='all')
            for col in df_to_push.columns:
                sample_series = df_to_push[col].dropna()
                if not sample_series.empty:
                    sample_val = sample_series.iloc[0]
                    is_numpy_dt = isinstance(sample_val, np.datetime64)
                    is_pandas_dt = isinstance(sample_val, pd.Timestamp)
                    is_python_dt = isinstance(sample_val, datetime.datetime)
                    if is_numpy_dt or is_pandas_dt or is_python_dt:
                        # Di dalam loop konversi, ubah jadi:
                            df_to_push[col] = df_to_push[col].apply(
                                lambda x: x.strftime('%Y-%m-%d %H:%M:%S') if pd.notnull(x) and hasattr(x, 'strftime') else None
    #                    ^^^^^^^^^^^^^^^^^^^ 
    # Format ini TIDAK mengandung .%f (mikrodetik), jadi aman untuk DATETIME biasa!
)
                if pd.api.types.is_datetime64_any_dtype(df_to_push[col]):
                    df_to_push[col] = df_to_push[col].apply(
                        lambda x: x.strftime('%Y-%m-%d %H:%M:%S') if pd.notnull(x) else None
                    )
            # ================================================================
            
            # Siapkan query
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Langsung convert ke list of tuples (tanpa ribet cleaning isna lagi, karena sudah aman)
            # Tapi kita tetap bersihkan kemungkinan ada NaN/None di kolom lain (angka/string)
            raw_data = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN", ""] else x for x in row) 
                for row in raw_data
            ]
            
            total_rows = len(clean_data_tuples)
            actual_inserted_total = 0
            db_warnings = []
            
            # 🔥 SISTEM AUTO-BATCHING (CHUNKING) 🔥
            # Loop memotong data menjadi bagian-bagian kecil agar MySQL tidak tersedak
            for i in range(0, total_rows, batch_size):
                chunk = clean_data_tuples[i : i + batch_size]
                cursor.executemany(insert_query, chunk)
                
                # Hitung data yang berhasil masuk pada batch ini
                chunk_inserted = max(0, cursor.rowcount)
                actual_inserted_total += chunk_inserted
                
                # Jika ada yang ter-skip di batch ini, tangkap errornya (maksimal simpan 3 per tabel)
                if chunk_inserted < len(chunk) and len(db_warnings) < 3:
                    cursor.execute("SHOW WARNINGS")
                    warnings_fetched = cursor.fetchall()
                    if warnings_fetched:
                        for w in warnings_fetched:
                            w_msg = f"MySQL Warning: {w['Message']}"
                            if w_msg not in db_warnings:
                                db_warnings.append(w_msg)
                            if len(db_warnings) >= 3:
                                break
                                
                # Commit per batch agar memori stabil
                db_connection.commit()
            
            # Evaluasi Status Akhir Tabel
            if actual_inserted_total == total_rows:
                status_flag = 'success'
                msg = f'✓ {table_name}: SEMPURNA! {total_rows}/{total_rows} baris sukses masuk database.'
            else:
                status_flag = 'partial_warning'
                msg = f'⚠️ {table_name}: TER-SKIP! Dikirim {total_rows} baris, tapi yang masuk DB HANYA {actual_inserted_total} baris.'

            results[table_name] = {
                'status': status_flag, 
                'msg': msg,
                'warnings': db_warnings
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'msg': f'✗ {table_name}: Gagal total saat eksekusi insert - Alasan: {e}',
                'warnings': []
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    print("🟢 TABEL YANG 100% SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') == 'success':
            print(f"  {res['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses sempurna)")

    print("\n🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):")
    failed_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') in ['failed', 'partial_warning', 'not_found', 'empty']:
            print(f"  {res['msg']}")
            
            # Cetak alasan dari MySQL (Dibatasi 3 agar tidak merusak tampilan Jupyter)
            if res.get('warnings'):
                for w_msg in res['warnings']:
                    print(f"      -> 🕵️ {w_msg}")
                    
            failed_exist = True
            
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada data yang terbuang.")
            
    print("================================================================================\n")

    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            res = results[table_name]
            
            if res['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name].head(3))
                print("-" * 80)
                
            elif res['status'] in ['failed', 'partial_warning']:
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Pesan Sistem: {res['msg']}")
                print("-" * 50)
                print("Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):")
                display(tables_data[table_name].head(5))
                print(f"\nTipe data Pandas untuk tabel '{table_name}':")
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

## Output

In [17]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL
# ================================================================================
results_fase_4 = insert_data_with_preview_and_skip_v2(
    db_connection=db_future, 
    cursor=cursor_future, 
    tables_data=data_siap_insert,       # <--- Menggunakan data yang sudah di-mapping otomatis
    ordered_list=master_urutan_insert   # <--- Menggunakan urutan master buatanmu
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG 100% SUKSES MASUK:
  ✓ izin_karyawan: SEMPURNA! 957/957 baris sukses masuk database.
  ✓ verifikasi_izin: SEMPURNA! 2107/2107 baris sukses masuk database.
  ✓ verifikasi_absensi: SEMPURNA! 11/11 baris sukses masuk database.
  ✓ karyawan_resign: SEMPURNA! 51/51 baris sukses masuk database.
  ✓ jadwal: SEMPURNA! 556/556 baris sukses masuk database.
  ✓ jadwal_hari: SEMPURNA! 982/982 baris sukses masuk database.
  ✓ jadwal_detail: SEMPURNA! 17312/17312 baris sukses masuk database.
  ✓ jadwal_pengajar: SEMPURNA! 650/650 baris sukses masuk database.
  ✓ catatan_kelas: SEMPURNA! 13733/13733 baris sukses masuk database.
  ✓ catatan_kelas_tag: SEMPURNA! 1067/1067 baris sukses masuk database.

🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):
  ⚠️ absensi: TER-SKIP! Dikirim 13444 baris, tapi yang masuk DB HANYA 6410 baris.
      -> 🕵️ MySQL Warning:

,id_izin,id_karyawan,jenis_izin,tanggal_mulai,tanggal_selesai,waktu_mulai,waktu_selesai,keterangan_izin,dokumen_lampiran,created_at
0,1,4,Ijin,2023-11-10,2023-11-10,15:30:00,17:00:00,Al muslin ekskul,None,2023-11-13 07:34:57
1,2,4,Lembur,2023-11-13,2023-11-13,07:00:00,08:30:00,pengganti Al muslim,None,2023-11-13 07:35:47
2,3,5,Lembur,2023-11-26,2023-11-26,16:00:00,18:00:00,Mengganti 2 jam kerja Senin 27 November 2023 j...,1701093214_10eabbf62174540924e1.jpeg,2023-11-27 20:53:34


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: VERIFIKASI_IZIN]
--------------------------------------------------


,id_verifikasi_izin,id_izin,status_verifikasi_izin,catatan_verifikator,id_division,created_at
0,164,1,Diajukan,Tidak ada catatan,3.0,2026-06-17 12:04:59.808546
1,165,2,Diajukan,Tidak ada catatan,3.0,2026-06-17 12:04:59.808546
2,167,1,Disetujui,,NaN,2026-06-17 12:04:59.808546


--------------------------------------------------------------------------------

🚨 [🔴 DIAGNOSTIK TABEL ERROR: ABSENSI] 🚨
Pesan Sistem: ⚠️ absensi: TER-SKIP! Dikirim 13444 baris, tapi yang masuk DB HANYA 6410 baris.
--------------------------------------------------
Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):


...


Tipe data Pandas untuk tabel 'jadwal_siswa':
id_jadwal_siswa                        int64
id_jadwal                              int64
id_siswa                              object
tanggal_mulai                         object
tanggal_keluar                        object
tanggal_aktif                 datetime64[ns]
tambahan_sesi                          int64
tambahan_keterangan                   object
status_keluar                          int64
is_acc_rapor                           int64
status_ketuntasan                     object
catatan_ketuntasan_guru               object
catatan_ketuntasan_admin              object
ketuntasan_diperbarui_oleh            object
ketuntasan_diperbarui_pada            object
dtype: object
--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: CATATAN_KELAS]
--------------------------------------------------


,id_jadwal,id_jadwal_detail,catatan_kelas,topik_diskusi,tanggal_konfirmasi,hasil_konfirmasi,id_karyawan
0,7,1231,1. Bya ijin tidak hadir karena masih perjalana...,,2026-06-24 02:56:54.434834,,None
1,3,721,Kelas berjalan lancar. Valencia bisa mengikuti...,,2026-06-24 02:56:54.434834,,None
2,9,1171,Elycia didn't come. Harits and Kinan came 15 m...,,2026-06-24 02:56:54.434834,,None


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: CATATAN_KELAS_TAG]
--------------------------------------------------


,id_ck,id_topik_diskusi
0,1439,1
1,1666,4
2,1666,4


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁
